# 2.6 — Partitioning, and the Non-Neutral Zero Value

**Chapter 2, sections 2.6.3 and 2.7** (*The Non-Neutral Zero Value*, *Partitioning*).

**The question this notebook answers:** the same data, the same functions, three different
answers. Which is right, and why did Spark give the other two?

A partition is the unit of parallelism, and for most of a Spark program the partition count is
a performance concern and nothing more. This notebook is about the case where it stops being
one and starts changing the *answer* — silently, with no error, and differently on a laptop
than on a cluster.

It also covers the mechanics the chapter needs alongside it: `glom`, `repartition`,
`coalesce`, and custom partitioners.

Covers **Exercise 4** (four `aggregate` calls, at 1 and 4 partitions) and **Exercise 9** (build
1000 integers over 4 partitions, repartition to 8, coalesce to 2, and say which shuffled).

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, sections 2.6.3 and 2.7.
import os, tempfile
from pyspark.sql import SparkSession

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.6")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
print("Spark", spark.version, "on", sc.master)
print("cores available to this session (= slots):", sc.defaultParallelism)

Spark 4.2.0 on local[*]
cores available to this session (= slots): 18


## 1. Seeing the partitions

`getNumPartitions` reports how many there are; `glom` collapses each partition into a list, so
that the layout becomes visible. `glom` is a debugging tool — it is implemented in terms of
`collect`, so it must never be pointed at production-scale data — but partitioning is otherwise
invisible until it causes a problem, and this is how to look at it.

In [2]:
x = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 5)
print("partitions :", x.getNumPartitions())
print("layout     :", x.glom().collect())

partitions : 5


layout     : [[1, 2], [3, 4], [5, 6, 7, 8], [9, 10], [11, 12, 13, 14]]


> **That layout is not the even split you might expect, and the reason is worth knowing.**
>
> Spark sliced the data into 2, 2, 4, 2, 4 rather than 2, 3, 3, 3, 3. It did not slice the
> *elements*: PySpark first serializes a Python list into **batches**, with
> `batchSize = min(len(c) // numSlices, 1024)` — here `14 // 5 = 2` — and the JVM then divides
> those **seven batches** among the five partitions. Seven does not divide by five either, so
> the unevenness moves up a level and is magnified by the batch size.
>
> Passing a `range` instead takes a different code path that computes exact boundaries per
> partition, and gives the even split. Compare the two below. Neither is wrong; the point is
> that **`parallelize` does not promise an even layout**, and if you are relying on one, check.

In [3]:
print("from a list  :", sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14], 5)
                          .glom().collect())
print("from a range :", sc.parallelize(range(1, 15), 5).glom().collect())

from a list  : [[1, 2], [3, 4], [5, 6, 7, 8], [9, 10], [11, 12, 13, 14]]
from a range : [[1, 2], [3, 4, 5], [6, 7, 8], [9, 10, 11], [12, 13, 14]]


## 2. `repartition` versus `coalesce`

`repartition(n)` sets the count to exactly *n*, up or down, with a **full shuffle**: every
record is rehashed and redistributed. It spreads the data far more evenly than it was, though
hashing guarantees nothing exactly.

`coalesce(n)` can only **reduce** the count, and does so **without a full shuffle**: each new
partition simply absorbs several existing ones. No record is rehashed and nothing crosses the
network — which makes it cheap, and makes the result potentially uneven.

In [4]:
x = sc.parallelize([1, 2, 3, 4, 5], 2)
print("start           :", x.glom().collect())
print("repartition(3)  :", x.repartition(3).glom().collect(), " full shuffle")
print("coalesce(1)     :", x.coalesce(1).glom().collect(), "     no shuffle")

print("\nNote the empty partition that repartition(3) produced.  Hashing five records into")
print("three buckets is not obliged to fill all three; 'even' means 'as even as a hash")
print("makes it', not 'equal'.")

start           : [[1, 2], [3, 4, 5]]


repartition(3)  : [[1, 2], [3, 4, 5], []]  full shuffle


coalesce(1)     : [[1, 2, 3, 4, 5]]      no shuffle

Note the empty partition that repartition(3) produced.  Hashing five records into
three buckets is not obliged to fill all three; 'even' means 'as even as a hash
makes it', not 'equal'.


### Exercise 9

> *Build an RDD of 1000 integers with 4 partitions and show the layout with `glom()`.
> Then (a) `repartition` to 8 and show the layout; (b) `coalesce` to 2 and show the layout.
> Explain the difference in the resulting distributions, and say which operation caused a
> shuffle.*

Printing a thousand integers three times is unreadable, so the cell shows partition **sizes**,
which is what the question is actually about.

In [5]:
def sizes(rdd):
    return [len(p) for p in rdd.glom().collect()]

base = sc.parallelize(range(1000), 4)
print(f"start          {base.getNumPartitions()} partitions  sizes {sizes(base)}")

up = base.repartition(8)
print(f"repartition(8) {up.getNumPartitions()} partitions  sizes {sizes(up)}")

down = base.coalesce(2)
print(f"coalesce(2)    {down.getNumPartitions()} partitions  sizes {sizes(down)}")

print("\n(a) repartition(8) SHUFFLED.  It is the only way to increase the partition count:")
print("    a new partition has to be filled from records that live somewhere else.")
print("    The sizes come out near-equal because every record was rehashed.")
print("(b) coalesce(2) did NOT shuffle.  Each new partition absorbed two existing ones,")
print("    so the sizes are the sums of the originals -- even here only because the")
print("    originals were even.  Feed it uneven input and it hands back uneven output.")
print("\nThe canonical use of coalesce is at the END of a job, when a filter has left a")
print("large number of nearly-empty partitions and the output should be a few files")
print("rather than hundreds of tiny ones.")

start          4 partitions  sizes [250, 250, 250, 250]


repartition(8) 8 partitions  sizes [130, 120, 140, 120, 120, 120, 130, 120]
coalesce(2)    2 partitions  sizes [500, 500]

(a) repartition(8) SHUFFLED.  It is the only way to increase the partition count:
    a new partition has to be filled from records that live somewhere else.
    The sizes come out near-equal because every record was rehashed.
(b) coalesce(2) did NOT shuffle.  Each new partition absorbed two existing ones,
    so the sizes are the sums of the originals -- even here only because the
    originals were even.  Feed it uneven input and it hands back uneven output.

The canonical use of coalesce is at the END of a job, when a filter has left a
large number of nearly-empty partitions and the output should be a few files
rather than hundreds of tiny ones.


## 3. How records are assigned

By default Spark hash-partitions pair RDDs: a record's index is `hash(key) % numPartitions`.
Records with the same key therefore always land together, which is what makes any by-key
operation possible at all.

PySpark uses its own `portable_hash` rather than Python's built-in `hash`, and fixes
`PYTHONHASHSEED` on its workers. Python randomizes string hashing per process; if two worker
processes disagreed about a key's hash, equal keys would scatter across different partitions
and every by-key operation would silently break.

In [6]:
import os
from pyspark.rdd import portable_hash

pairs = sc.parallelize([("Alice", 1), ("Bob", 2), ("Catherine", 3), ("Al", 4)], 3)
by_hash = pairs.partitionBy(3)          # the default partitioner
print("hash partitioning:")
for i, part in enumerate(by_hash.glom().collect()):
    print(f"  partition {i}: {part}")

hash partitioning:


  partition 0: [('Catherine', 3)]
  partition 1: [('Alice', 1), ('Al', 4)]
  partition 2: [('Bob', 2)]


In [7]:
# Try to reproduce the arithmetic here, in the driver.  This fails, and the failure IS
# the footnote: Spark fixes PYTHONHASHSEED on its WORKERS, and this notebook's driver
# process was started without it.
try:
    portable_hash("Alice")
except Exception as e:
    print(f"in the driver: {type(e).__name__}")
    print(f"  {str(e).splitlines()[0]}")

print(f"\n  driver PYTHONHASHSEED = {os.environ.get('PYTHONHASHSEED')!r}")

in the driver: PySparkRuntimeError
  [PYTHON_HASH_SEED_NOT_SET] Randomness of hash of string should be disabled via PYTHONHASHSEED.

  driver PYTHONHASHSEED = None


In [8]:
# On the executors it works, because Spark set the seed there.  Asking each partition
# to report the seed it is running under, and the hash it computes, shows both facts
# at once -- and confirms the assignment above was not an accident of this run.
def describe(idx, records):
    seed = os.environ.get("PYTHONHASHSEED")
    for key, _ in records:
        yield (idx, key, portable_hash(key) % 3, seed)

rows = pairs.partitionBy(3).mapPartitionsWithIndex(describe).collect()

print(f"{'lands in':>9s}{'key':>12s}{'hash % 3':>10s}{'worker PYTHONHASHSEED':>24s}")
for idx, key, h, seed in sorted(rows, key=lambda r: r[0]):
    print(f"{idx:>9}{key:>12s}{h:>10}{str(seed):>24s}")

print("\nThe 'lands in' and 'hash % 3' columns agree for every key, which is the whole")
print("guarantee: equal keys hash equally, so they land together.  If two workers")
print("disagreed about a string's hash -- which is what Python's per-process hash")
print("randomization would cause -- equal keys would scatter across partitions and every")
print("by-key operation in Spark would silently return wrong answers.  Fixing the seed")
print("on the workers is what prevents that, and PySpark refuses to compute the hash at")
print("all where the seed is not fixed rather than let you get it subtly wrong.")

 lands in         key  hash % 3   worker PYTHONHASHSEED
        0   Catherine         0                       0
        1       Alice         1                       0
        1          Al         1                       0
        2         Bob         2                       0

The 'lands in' and 'hash % 3' columns agree for every key, which is the whole
guarantee: equal keys hash equally, so they land together.  If two workers
disagreed about a string's hash -- which is what Python's per-process hash
randomization would cause -- equal keys would scatter across partitions and every
by-key operation in Spark would silently return wrong answers.  Fixing the seed
on the workers is what prevents that, and PySpark refuses to compute the hash at
all where the seed is not fixed rather than let you get it subtly wrong.


In [9]:
# A custom partitioner: any function from key to integer.  This one partitions by the
# length of the name, which is the chapter's example -- and a deliberately bad one.
def custom_partitioner(key):
    return len(key)

custom = pairs.partitionBy(3, custom_partitioner)
print("partitioned by len(name):")
for i, part in enumerate(custom.glom().collect()):
    print(f"  partition {i}: {part}")

print("\nCustom partitioning can CORRECT skew, when hashing happens to send too many")
print("records to one partition, and can make repeated joins on the same key cheap by")
print("co-partitioning both sides once.  It can also INTRODUCE skew, which is what")
print("partitioning names by length does: most names are of similar length, so most")
print("records land in a few partitions.  Chapter 3 covers Spark's automatic remedy.")

partitioned by len(name):
  partition 0: [('Bob', 2), ('Catherine', 3)]
  partition 1: []
  partition 2: [('Alice', 1), ('Al', 4)]

Custom partitioning can CORRECT skew, when hashing happens to send too many
records to one partition, and can make repeated joins on the same key cheap by
co-partitioning both sides once.  It can also INTRODUCE skew, which is what
partitioning names by length does: most names are of similar length, so most
records land in a few partitions.  Chapter 3 covers Spark's automatic remedy.


## 4. The zero value that is not a zero

Now the part that changes answers rather than performance.

`aggregate`, `fold` and `aggregateByKey` all take a zero value, and that zero value is applied
**once per partition**. `aggregate` applies it **once more** when the per-partition results are
combined on the driver. If the zero is genuinely neutral for the operation — `0` for addition,
`1` for multiplication, the empty set for union — this is harmless. If it is not, the result
depends on the number of partitions.

The input below is eight 2s. `seq` accumulates a running (sum, count); `comb` adds two such
pairs together.

In [10]:
rdd = sc.parallelize([2, 2, 2, 2, 2, 2, 2, 2], 8)

seq = lambda acc, v: (acc[0] + v, acc[1] + 1)      # (running sum, running count)
comb = lambda a, b: (a[0] + b[0], a[1] + b[1])

print("Neutral zero (0, 0) -- correct whatever the partitioning:")
for n in (1, 4, 8, 10):
    print(f"  {n:>2} partitions : {rdd.repartition(n).aggregate((0, 0), seq, comb)}")

print("\nThe true answer is (16, 8): eight values of 2.")

Neutral zero (0, 0) -- correct whatever the partitioning:


   1 partitions : (16, 8)
   4 partitions : (16, 8)


   8 partitions : (16, 8)


  10 partitions : (16, 8)

The true answer is (16, 8): eight values of 2.


In [11]:
print("NON-neutral zero (1, 1) -- the extra 1s are added once per partition,")
print("and once MORE in the final combine on the driver:\n")
for n in (1, 4, 8, 10):
    got = rdd.repartition(n).aggregate((1, 1), seq, comb)
    print(f"  {n:>2} partitions : {got}   "
          f"(= (16, 8) + {n + 1} x (1, 1))")

print("\nFour different answers, from the same data and the same two functions.")
print("Nothing malfunctioned.  Spark performed exactly the computation requested;")
print("the requested computation was wrong.")

NON-neutral zero (1, 1) -- the extra 1s are added once per partition,
and once MORE in the final combine on the driver:



   1 partitions : (18, 10)   (= (16, 8) + 2 x (1, 1))
   4 partitions : (21, 13)   (= (16, 8) + 5 x (1, 1))


   8 partitions : (25, 17)   (= (16, 8) + 9 x (1, 1))
  10 partitions : (27, 19)   (= (16, 8) + 11 x (1, 1))

Four different answers, from the same data and the same two functions.
Nothing malfunctioned.  Spark performed exactly the computation requested;
the requested computation was wrong.


### The same thing with `treeAggregate`, which gives different wrong answers

`treeAggregate` combines the partial results in a tree of intermediate stages **on the
executors** rather than all at once on the driver. It therefore applies the zero value once per
partition but **not** in the final combine — so with a non-neutral zero, even the choice of
aggregation *method* changes the result.

In [12]:
print(f"{'partitions':>11s}{'aggregate':>16s}{'treeAggregate':>18s}{'difference':>13s}")
for n in (1, 4, 8, 10):
    a = rdd.repartition(n).aggregate((1, 1), seq, comb)
    t = rdd.repartition(n).treeAggregate((1, 1), seq, comb)
    print(f"{n:>11}{str(a):>16s}{str(t):>18s}{str((a[0] - t[0], a[1] - t[1])):>13s}")

print("\nThe difference is exactly one application of the zero value, every time: the one")
print("aggregate() performs on the driver and treeAggregate() does not.")

 partitions       aggregate     treeAggregate   difference


          1        (18, 10)           (17, 9)       (1, 1)


          4        (21, 13)          (20, 12)       (1, 1)


          8        (25, 17)          (24, 16)       (1, 1)


         10        (27, 19)          (26, 18)       (1, 1)

The difference is exactly one application of the zero value, every time: the one
aggregate() performs on the driver and treeAggregate() does not.


### The rule

**The zero value must be an identity element for the combine function.** If
`comb(zero, x) == x` for every `x`, the computation is correct and the partition count cannot
affect it. Otherwise the answer depends on the partition count — which *will* change when the
code moves from a laptop to a cluster, and the computed values will change with it.

The cell below is that rule as a test you can actually run.

In [13]:
def is_identity(zero, comb, samples):
    """comb(zero, x) == x for every sample x?"""
    return all(comb(zero, x) == x for x in samples)

samples = [(16, 8), (0, 0), (3, 1), (100, 25)]
for zero in [(0, 0), (1, 1), (0, 1)]:
    ok = is_identity(zero, comb, samples)
    print(f"  zero = {str(zero):8s} identity? {'YES' if ok else 'NO '}"
          f"   -> {'safe' if ok else 'the answer will depend on the partition count'}")

  zero = (0, 0)   identity? YES   -> safe
  zero = (1, 1)   identity? NO    -> the answer will depend on the partition count
  zero = (0, 1)   identity? NO    -> the answer will depend on the partition count


### Exercise 4

> *For each call, state whether the zero value is an identity element; and if it is not, give
> the answer you would get with 1 partition and with 4 partitions over the input
> `[1, 1, 1, 1]`.*

In [14]:
data = [1, 1, 1, 1]
cases = [
    ("(a) aggregate(0,  a+v, a+b)", 0,  lambda a, v: a + v, lambda a, b: a + b),
    ("(b) aggregate(1,  a+v, a+b)", 1,  lambda a, v: a + v, lambda a, b: a + b),
    ("(c) aggregate(1,  a*v, a*b)", 1,  lambda a, v: a * v, lambda a, b: a * b),
    ("(d) aggregate([], a+[v], a+b)", [], lambda a, v: a + [v], lambda a, b: a + b),
]

print(f"{'call':32s}{'identity?':>11s}   {'1 partition':<16s}{'4 partitions':<16s}")
for label, zero, sq, cb in cases:
    probes = [0, 1, 5] if not isinstance(zero, list) else [[], [1], [1, 2]]
    identity = all(cb(zero, x) == x for x in probes)
    one = sc.parallelize(data, 1).aggregate(zero, sq, cb)
    four = sc.parallelize(data, 4).aggregate(zero, sq, cb)
    print(f"{label:32s}{'YES' if identity else 'NO':>11s}   "
          f"{str(one):<16s}{str(four):<16s}")

print("\n(a) 0 is the identity for +, so 4 = 4 whatever the partitioning.")
print("(c) 1 is the identity for *, so 1 = 1 likewise.")
print("(b) 1 is NOT the identity for +: one extra 1 per partition, plus one on the driver.")
print("(d) [] IS the identity for list concatenation -- the answer is stable, and the")
print("    ORDER is not, because it follows the partition order.  A stable value and an")
print("    unstable order is its own trap if anything downstream depends on the order.")

call                              identity?   1 partition     4 partitions    


(a) aggregate(0,  a+v, a+b)             YES   4               4               
(b) aggregate(1,  a+v, a+b)              NO   6               9               
(c) aggregate(1,  a*v, a*b)             YES   1               1               
(d) aggregate([], a+[v], a+b)           YES   [1, 1, 1, 1]    [1, 1, 1, 1]    

(a) 0 is the identity for +, so 4 = 4 whatever the partitioning.
(c) 1 is the identity for *, so 1 = 1 likewise.
(b) 1 is NOT the identity for +: one extra 1 per partition, plus one on the driver.
(d) [] IS the identity for list concatenation -- the answer is stable, and the
    ORDER is not, because it follows the partition order.  A stable value and an
    unstable order is its own trap if anything downstream depends on the order.


## 5. A forward reference: partitioning a DataFrame

The DataFrame API has the same three ideas under different names, and chapter 3 develops them.
They are shown here in three lines only so that the correspondence is visible while the RDD
version is fresh.

In [15]:
data = [("Alice", "IT", 34), ("Bob", "HR", 45), ("Tom", "HR", 29), ("Catherine", "Finance", 28),
        ("Harry", "Finance", 45), ("Alvin", "IT", 10), ("Ana", "HR", 32), ("Bea", "Finance", 19)]
df = spark.createDataFrame(data, ["name", "dept", "age"])

def show_df_partitions(d, label):
    parts = d.rdd.glom().collect()
    print(f"{label} -> {len(parts)} partitions")
    for i, p in enumerate(parts):
        if p:
            print(f"   {i}: {[r['name'] for r in p]}")

show_df_partitions(df.repartition(3, "dept"), "repartition(3, 'dept')   hash by column")
show_df_partitions(df.repartitionByRange(3, "age"), "repartitionByRange(3, 'age')  by value range")
show_df_partitions(df.coalesce(1), "coalesce(1)              merge, no shuffle")

repartition(3, 'dept')   hash by column -> 3 partitions
   0: ['Alice', 'Alvin']
   1: ['Bob', 'Tom', 'Ana']
   2: ['Catherine', 'Harry', 'Bea']
repartitionByRange(3, 'age')  by value range -> 3 partitions
   0: ['Catherine', 'Alvin', 'Bea']
   1: ['Alice', 'Tom', 'Ana']
   2: ['Bob', 'Harry']
coalesce(1)              merge, no shuffle -> 1 partitions
   0: ['Alice', 'Bob', 'Tom', 'Catherine', 'Harry', 'Alvin', 'Ana', 'Bea']


`repartition(n, col)` is hash partitioning, exactly as `partitionBy` is for a pair RDD.
`repartitionByRange(n, col)` is the DataFrame form of what `sortByKey` does internally: it
samples the column to choose boundaries, so each partition holds a contiguous range of values.
`coalesce` behaves as it does on an RDD. Chapter 3 returns to all of this, together with the
skew handling that Adaptive Query Execution performs automatically.

## Conclusion

**Partitioning is the performance control a newcomer can most readily apply.** The partition
count sets the ceiling on parallelism — one task per partition, one task at a time per core —
and the partition *sizes* decide whether that parallelism is any use. Far below the number of
cores leaves cores idle; far above it pays scheduling overhead for parallelism that is not
there. The Spark tuning guide's rule of thumb is two to three tasks per core.

**`repartition` shuffles and `coalesce` does not**, and that is the whole difference between
them: one buys evenness with a network round trip, the other takes what it is given for free.

**And the zero value is not a formality.** It is the one place in this chapter where getting it
wrong produces a *wrong answer* rather than a slow one or an exception. The test is a single
line — `comb(zero, x) == x` — and it is worth running in your head every time an `aggregate`,
a `fold`, or an `aggregateByKey` is written.

**Next.** Notebook 2.7 makes the shuffle itself visible: where stage boundaries fall, why one
action is one job, and what caching changes.